# SDD编程

## 阶段一：准备阶段

### 1. 项目初始化

#### 1.1 根目录初始化 Git

在项目根目录运行：git init

- 一个隐藏的 .git 文件夹 

#### 1.2 根目录初始化 mattpocock-skills

在根目录执行：/mattpocock Skills:setup Matt Pocock Skills


## 阶段二：SDD 开发流程

### 1. mattpocock-skills 流程

- /Mattpocock Skills:grill-with-docs
做一个全栈项目，前后端放入单独的文件夹
- /Mattpocock Skills:to-spec
- /Mattpocock Skills:to-tickets
- /Mattpocock Skills:implement-spec
>- /Mattpocock Skills:retro

### 2. 各任务顺序

- 聊需求（确定技术栈）
- 实现各功能
- 系统界面与交互
- 检查一下有没有没用的代码和文档
>- 更新.gitignore 、README.md 、目录.md 后提交 git

### 3. 提交到 GitHub（推荐 HTTPS）

#### 3.1 GitHub 建立新仓库

登陆 GitHub 建立新仓库( new repository )

#### 3.2 复制 HTTPS 地址

```text
https://github.com/你的用户名/项目名.git
```

### 3.3 关联本地仓库

```bash
git remote add origin https://github.com/你的用户名/项目名.git
git remote -v  # 查看远程仓库
```

### 3.4 第一次推送

先用 `git branch` 确认分支名。当前分支为 `main` 时执行：

```bash
git branch
git push -u origin main

git push    # 后续只需要这样
```

## 阶段三：服务器部署与上线

### 1. 准备云服务器

云服务器平台示例：阿里云、百度云。

需要准备：
- 云服务器或轻量应用服务器，操作系统选择 Ubuntu 24.04 LTS。
- 要有公网 IP。

进入云平台的安全组或防火墙页面，配置入站连接：

| TCP 端口 | 用途 |
| --- | --- |
| 22 | SSH 登录 |
| 80 | HTTP 访问及常见证书验证 |

### 2. 登录服务器、准备部署用户

#### 2.1 连接服务器

在本机终端输入：

```bash
ssh 登录用户名@服务器公网IP
```

使用云平台提供的账号和认证方式登录

#### 2.2 准备普通部署用户

如果已有具备 sudo 权限的普通用户，可以直接使用。如果当前以 root 登录，需要创建普通用户时执行：

```bash
adduser 部署用户名          # 创建用户
usermod -aG sudo 部署用户名 # 给用户添加 sudo 权限
su - 部署用户名             # 切换用户
```

#### 2.3 解决 SSH 因空闲自动断连

在目录 `C:\Users\你的Windows用户名\.ssh\config` 添加以下内容，文件不存在时新建 `config`文件（无后缀）

```text
Host *
  ServerAliveInterval 15
  ServerAliveCountMax 3
```

客户端每 15 秒发送一次保活消息，连续 3 次未收到回复时断开。

### 3. 安装所需工具

#### 3.1 安装基础工具和 Nginx

在服务器终端执行：

```bash
sudo apt update
sudo apt install -y git curl ca-certificates nginx
sudo systemctl enable --now nginx
systemctl status nginx --no-pager
nginx -v
git --version
```

`enable --now` 表示立即启动，并设置服务器开机后自动启动。

#### 3.2 安装其他工具

询问 AI 为服务器安装其他工具

>交互提示词示例：

```text
请根据项目 README.md 中确定的技术栈和依赖版本，为 Ubuntu 24.04 LTS 服务器提供环境安装步骤。

服务器通过 SSH 操作，我会手动执行你给出的命令。

1. 先给出检查所需工具的命令。
2. 我返回检查结果后，只为缺少或版本不兼容的工具提供安装命令，优先使用官方安装方式。
3. 标明每条命令的执行位置、用途，以及是否需要 sudo。
4. 安装后给出版本和可执行文件路径的检查命令，供后续配置 systemd 使用。

尽量复用已有环境，避免影响其他项目。
```

### 4. 克隆项目并安装依赖

#### 4.1 配置 GitHub 账号 SSH 密钥

（Deploy Key 作为以后生产部署的参考）

以部署用户登录服务器，先检查是否已有密钥：

```bash
ls -la ~/.ssh
```

目录不存在属于正常情况。需要创建新密钥时执行：

```bash
ssh-keygen -t ed25519 -C "你的邮箱"
```

直接回车选择默认选择，查看默认路径下的公钥：

```bash
cat ~/.ssh/id_ed25519.pub
```

复制以 `ssh-ed25519` 开头的完整公钥，在 GitHub 账号中进入：

```text
GitHub → Settings → SSH and GPG keys → New SSH key
```

填写标题，添加公钥。

验证连接：

```bash
ssh -T git@github.com
```

#### 4.2 克隆项目

在服务器执行：

```bash
cd ~
git clone git@github.com:你的用户名/项目名.git
cd 项目名
```

#### 4.3 按照锁文件安装项目依赖

从项目根目录开始：

```bash
cd frontend
npm ci
cd ../backend
uv sync --locked
cd ..
```

### 5. 配置生产环境变量、数据库和数据目录

#### 5.1 准备生产配置

配置所有`.env*example`文件

#### 5.2 配置前端生产后端地址

这份笔记采用同一域名访问前端和 API，由 Nginx 将 `/api/` 请求转发到 FastAPI。

如果项目沿用准备阶段的写法，即把 `API_URL` 与 `/api/...` 拼接，则可以在服务器 `frontend/.env.production` 中配置网站的完整来源地址，不附加 `/api` 或末尾斜杠。

React + Vite：

```dotenv
VITE_API_URL=https://你的域名
```

Next.js：

```dotenv
NEXT_PUBLIC_API_URL=https://你的域名
```

项目也可以统一使用 `/api/...` 相对路径；是否需要上述变量，以实际请求代码为准，避免重复拼接 `/api`。暂时通过公网 IP 验证 HTTP 时，可先使用 `http://服务器公网IP`，切换 HTTPS 域名后再更新配置并重新构建。

浏览器中的 `127.0.0.1` 指向访问者自己的电脑，不能把本地后端地址原样用于正式网站。`VITE_` 和 `NEXT_PUBLIC_` 开头的变量会提供给浏览器，不要放模型 API 密钥等秘密。

生产地址应在前端构建前配置；Vite 和 Next.js 的浏览器公开环境变量通常会写入构建产物，修改后需要重新构建并发布。不要把本地调试用的 `.env.local` 原样复制到服务器，以免覆盖生产配置。

#### 5.3 准备数据库和持久化目录

- SQLite：确定数据库文件的绝对路径，并让后端服务用户能够读写数据库所在目录。
- PostgreSQL：准备数据库、应用账号和连接配置，按设计授予权限。
- 为数据库文件、上传资料和备份准备持久化目录，避免放进会被构建、清理或替换的代码发布目录。
- 首次部署执行项目规定的初始化或迁移命令；已有数据时先备份，再执行必要的迁移，不重复执行会清空数据的初始化。

>交互提示词示例：

```text
请检查当前项目代码、README.md 和数据库设计，整理服务器生产配置与数据准备步骤。

列出实际需要的环境变量、用途、保存位置和加载方式；区分前端公开配置与后端私密配置，提供不含真实密钥的示例。检查前端 API 地址和 /api 路径的拼接方式，给出与 Nginx 同域转发匹配的配置。

根据当前数据库方案，给出数据库创建、账号权限、初始化或迁移命令，以及数据库、上传文件、备份的实际目录和文件权限。说明命令的执行目录，并区分首次部署和已有数据的情况。

确认后端服务运行用户能够访问所需数据。把不含秘密的操作说明记录到 docs/deployment.md。
```

### 6. 构建前端，启动并管理应用服务

目标：生产服务在 SSH 断开后继续运行，发生进程退出时可按配置恢复，并在服务器重启后自动启动。

#### 6.1A React + Vite

在服务器的 `frontend` 目录执行：

```bash
npm run build
```

默认构建产物在 `frontend/dist/`。第 7 步将其发布到静态网站目录，例如 `/var/www/项目名/`，由 Nginx 提供访问。若项目修改了输出目录，以实际配置为准。

Vite 的这种部署方式不需要常驻 Node 服务；`npm run dev` 和 `npm run preview` 不用作本笔记的生产网站服务。

#### 6.1B Next.js

先根据项目实际功能确定部署模式。

需要 Next.js 服务端能力时，在 `frontend` 目录构建：

```bash
npm run build
```

如果 `package.json` 中的 `start` 脚本是 `next start`，可以用下面的命令临时检查启动：

```bash
npm run start -- --hostname 127.0.0.1 --port 3000
```

正式运行交给第 6.3 步的 systemd 管理，再由 Nginx 转发请求。检查结束后按 `Ctrl+C` 停止临时进程，避免与正式服务争用端口。项目如果采用 standalone 输出，按该模式的实际入口和文件布局部署。

只有项目满足静态导出条件、且已配置静态导出时，才按静态文件部署；默认导出目录通常为 `out/`，该模式无需启动 Next.js 服务。

#### 6.2 启动 FastAPI 后端

以下示例适用于后端入口仍为 `backend/main.py`、应用变量为 `app` 的项目。在 `backend` 目录临时检查：

```bash
uv run --locked fastapi run main.py --host 127.0.0.1 --port 8000
```

入口发生变化时，以实际项目结构为准。确保当前启动方式已经加载第 5 步的后端配置；最终由 systemd 统一管理配置和启动。

在另一个服务器终端检查：

```bash
curl -f http://127.0.0.1:8000/api/health
```

应该返回 `{"status":"ok"}`。检查结束后按 `Ctrl+C` 停止临时进程；正式服务不使用 `fastapi dev` 或自动重载模式。

#### 6.3 配置 systemd 服务

FastAPI 需要常驻服务；采用 Node 服务模式的 Next.js 也需要。Vite 静态前端不需要创建 Node 服务。

>交互提示词示例：

```text
请根据当前项目和 Ubuntu 24.04 LTS 服务器环境，为应用生成可实际使用的生产构建、启动和 systemd 配置。

部署用户：【实际用户名】
项目绝对路径：【实际路径】
前端部署模式：【Vite 静态文件 / Next.js Node 服务 / Next.js 静态导出】
生产配置和数据目录：【第 5 步确认的路径】

要求：
1. 检查 package.json、后端入口和真实可执行文件路径，给出正确的构建与生产启动命令。
2. 为 FastAPI，以及确实需要的 Next.js Node 服务分别生成 systemd unit；设置 User、WorkingDirectory、ExecStart、实际配置文件的加载方式、失败后重启和开机启动。
3. ExecStart 使用明确的可执行文件路径，不依赖交互式终端的 PATH、手动激活虚拟环境或登录脚本；后端可直接使用项目 .venv 中的生产服务程序。
4. FastAPI 监听 127.0.0.1:8000，Next.js Node 服务监听 127.0.0.1:3000；端口已占用时先查明原因，再确定实际配置。
5. 说明文件保存位置、权限及各项配置含义，给出 daemon-reload、enable --now、status、restart、stop 和 journalctl 查看日志的完整命令，使用实际服务名。
6. 给出服务器内部的健康检查方法。检查完成后记录到 docs/deployment.md。
```

### 7. 配置 Nginx、域名和 HTTPS

#### 7.1 确定访问方式

- React + Vite：Nginx 提供构建后的静态文件；使用浏览器路径路由时，配置前端页面的回退规则。
- Next.js Node 服务：Nginx 将前端请求转发到本机 Next.js 服务。
- Next.js 静态导出：Nginx 提供导出的静态文件，并按实际导出目录结构处理路由。
- `/api/` 请求：转发到 FastAPI。当前后端路由包含 `/api` 前缀，例如 `/api/health`，因此转发时必须保留该前缀。

静态文件优先发布到 `/var/www/项目名/` 等专用目录，只给 Nginx 必需的目录访问和文件读取权限。应用代码、数据库和私密配置不作为静态文件公开。不统一修改整个家目录的权限，也不使用 `chmod 777`。

#### 7.2 配置 HTTP 和 Nginx

>交互提示词示例：

```text
请根据当前项目和服务器上的实际部署结果，教我一步步配置 Nginx。

服务器公网 IP：【填写】
域名：【填写；暂时没有则说明】
前端部署模式和构建产物位置：【填写】
FastAPI 内部地址：【例如 http://127.0.0.1:8000】
Next.js 内部地址：【仅 Node 服务模式填写】

请给出：
1. 需要创建或修改的配置文件绝对路径，说明现有配置的含义、修改内容及原因；保留服务器上其他网站的配置。
2. 与当前前端模式匹配的站点配置和发布命令；静态文件放在专用网站目录，处理实际需要的前端路由和静态资源访问。
3. /api/ 到 FastAPI 的转发配置。先检查后端路由，再确定 proxy_pass 写法，避免把 /api/health 错转成 /health；说明请求路径的实际变化。
4. 正确传递 Host、客户端地址和协议相关头部；检查后端对代理头的信任设置，仅信任实际代理。项目涉及流式 AI 输出或 WebSocket 时，配置相应的缓冲、超时或连接升级规则。
5. 配置文件启用方式，以及 nginx -t 校验、重载和排错日志命令。
6. 验证首页、静态资源、页面刷新和 /api/health 的方法。

如果前后端确实使用不同来源，再按实际生产地址配置 CORS。把最终方案记录到 docs/deployment.md。
```

修改后先校验，通过后再重载：

```bash
sudo nginx -t
sudo systemctl reload nginx
```

#### 7.3 配置域名和 HTTPS

在域名服务商处把域名解析到服务器公网 IP；检查已设置的 A/AAAA 记录是否都指向可用服务器。确认 80、443 端口能访问；按所选云服务区域和服务商要求完成必要的域名接入或备案流程。

本笔记按“域名解析 + Nginx + ACME 证书”的常见方式配置 HTTPS，具体安装命令以服务器当前环境为准。

>交互提示词示例：

```text
当前 Nginx 的 HTTP 站点已验证。请基于 Ubuntu 24.04 LTS 和现有 Nginx 配置，给出为【实际域名】配置 HTTPS 的操作步骤。

检查域名解析和端口可达性，选择与当前环境兼容的 Certbot 安装方式，说明证书申请、Nginx 配置、HTTP 跳转 HTTPS、自动续期及续期测试命令。

如果项目有登录会话，检查 HTTPS 下的 Cookie 配置；验证浏览器不再请求旧的 HTTP 或 localhost 地址。前端公开环境变量发生变化时，重新构建并发布；需要运行 Next.js 服务时重启对应服务。

记录证书位置、续期方式、实际验证结果和未完成事项到 docs/deployment.md。
```

证书配置完成后，可以执行续期演练，并确认自动续期任务已启用：

```bash
sudo certbot renew --dry-run
```

### 8. 线上验收

目标：确认用户通过真实网站地址可以完成主要任务，且数据和应用服务在重启后仍然正常。

至少验证：
- 从本机浏览器或另一台设备，通过最终 HTTPS 地址访问网站。
- 首页、静态资源、主要页面切换和页面刷新正常。
- `/api/health` 正常，实际业务接口可以调用。
- 按用户故事完成主要流程；真实模型调用、流式输出或上传等能力按项目范围验证。
- 新增和修改的数据可以再次读取，刷新页面和重启服务后仍然存在。
- 项目要求的登录、权限和数据隔离按验收标准工作。
- SSH 断开后应用仍运行；首次上线的可安排时段内，重启服务器并确认服务自动恢复、数据仍在。
- 生产构建、前后端日志和 Nginx 日志中没有未处理的部署错误。

使用可识别的测试数据，测试后只清理本次创建的数据。已有用户使用的网站，重启等操作安排在维护时段。

>交互提示词示例：

```text
请根据 docs/spec/MVP 中的用户故事与验收标准，以及 docs/deployment.md，对已部署的【实际网站地址】进行线上验收。

检查页面与 API、主要业务流程、真实 AI 服务、异常处理、数据持久化和需求中规定的访问权限，并验证 SSH 断开后服务持续运行。根据当前网站使用情况安排服务及服务器重启验证，确认自动启动和数据保留。

使用本次创建的测试数据，避免修改已有真实业务数据。记录检查方法、实际结果和日志线索；无法直接访问服务器或验证的部分，给我具体执行步骤，不要把未执行的检查写成通过。

发现问题时在当前 MVP 范围内修复，更新受影响的服务或构建产物，并重新验证相关流程。
把验收结果和未解决事项记录在 docs/deployment.md，确认是否达到当前版本的上线要求。
```

### 9. 更新发布、备份恢复和回退

#### 9.1 后续更新发布

先在开发环境完成修改和验收，再提交推送。服务器更新前记录当前代码版本、配置和可用构建产物，并备份受影响的数据。

在服务器项目目录内检查和拉取当前发布分支：

```bash
git status --short
git rev-parse HEAD
git pull --ff-only
```

如果有未提交改动、分支分歧，或当前采用标签/提交号部署，先按实际发布方式处理，不直接覆盖或强行拉取。

更新的实际顺序通常为：

1. 记录旧版本并备份，明确本次是否需要维护时段。
2. 更新到已验收的代码，按锁文件同步所需依赖。
3. 根据新版本要求准备配置；按迁移方案执行必要的数据库变更。
4. 重新构建前端，并发布新的静态文件或重启 Next.js 服务。
5. 重启受影响的后端服务；Nginx 配置有变化时先校验再重载。
6. 验证健康检查和主要业务流程，记录本次部署的提交号与结果。

代码、数据库和前端产物必须匹配同一发布版本。具体停机、迁移和切换顺序按项目兼容性确定；不要在数据库结构尚未就绪时切换不兼容的新代码。

#### 9.2 备份与恢复

按项目实际情况备份数据库、上传文件和恢复运行所需的配置。备份频率、保留时间和保存位置根据能接受的数据损失确定；至少保留一份服务器之外的受保护副本。

- SQLite：使用 SQLite 支持的备份方式，或按停写方案备份完整数据库；运行中的数据库尤其是 WAL 模式，不只随手复制一个 `.db` 文件。
- PostgreSQL：按恢复要求选择 `pg_dump` 等备份方式，并记录对应的恢复命令。
- 在独立数据库或隔离目录进行恢复演练，确认数据可读取、应用可运行，不覆盖正在使用的数据。

#### 9.3 发布失败时回退

保留可用的上一版本代码、依赖锁文件、构建产物和对应配置。发布失败时，按已记录的步骤恢复应用版本并重新验收。

如果新版本变更了数据库结构，需要先检查旧代码是否兼容；代码回退不会自动回退数据库。恢复旧数据库备份可能丢失备份之后的新数据，必须明确恢复范围和处理办法。

>交互提示词示例：

```text
请根据当前服务器部署方式，完善 docs/deployment.md 中的更新发布、备份恢复与失败回退流程。

要求：
1. 使用实际目录、服务名和命令，写清版本记录、数据备份、代码更新、锁文件依赖同步、环境配置、数据库迁移、前端构建与发布、服务重启及验收顺序。
2. 给出适合当前数据库和上传目录的备份方案，包括执行频率、保留时间、服务器之外的保存位置、访问权限和结果检查方式；涉及需要我提供的存储信息时明确列出。
3. 给出在隔离环境恢复备份并验证的步骤，确认备份实际可用。
4. 给出回退到上一可用版本的方法，区分应用回退与数据库恢复，并说明如何处理发布后的新增数据。
5. 记录日志位置和常用查看命令，给出适合当前规模的服务异常、磁盘空间和备份失败检查方式。
6. README.md 中添加部署文档入口；文档只记录操作方法和配置占位值，不记录真实密钥。

标明已实际验证和仍未验证的部分。不要把仅生成了命令当作已经完成发布或恢复演练。
```

完成标准：线上主要功能已经验收，服务能持续运行并在重启后恢复，数据持久化正常，备份恢复已验证，而且下一次更新和失败回退有可执行的说明。